# LLM-Assisted Intervention Identification — Telco Churn

**Reproducible experiment** for the Global Score and Path-Based Structural Importance (λ = 0.75) input workbooks.

The LLM identifies candidate interventions from causal signs and variable semantics. Rankings and importance scores are **not** used to decide the action. This notebook makes **one API request** for all distinct `(feature, sign)` pairs and writes both completed workbooks. It records an audit trail. No causal effect is claimed from the output.

**Security:** supply `OPENAI_API_KEY` through an environment variable or a hidden password prompt. 

**Model configuration:** `gpt-6-astra` through the Responses API. 

In [ ]:
# %pip install -U openai openpyxl nbformat

from pathlib import Path
from datetime import datetime, timezone
from getpass import getpass
import os, json, hashlib, time
from collections import defaultdict
from openpyxl import load_workbook
from openai import OpenAI
import re

DATA_DIR = Path.cwd()  # place the four input files beside the notebook
FILES = {
    "global": "Global Score - calculated - input LLM.xlsx",
    "path": "Path Based Structural Importance - calculated - 075 - input LLM.xlsx",
    "original": "Telco-Customer-Churn.xlsx",
    "encoded": "Telco-Churn-OneHot-MinMax.xlsx",
}
MODEL = "gpt-6-astra"  # verified as accessible by the researcher
# No temperature, top_p, or seed is supplied; compatibility is model-dependent.
OUTPUT_DIR = DATA_DIR / "llm_results"
OUTPUT_DIR.mkdir(exist_ok=True)
for key, name in FILES.items():
    if not (DATA_DIR / name).is_file():
        raise FileNotFoundError(f"Missing {key}: {DATA_DIR / name}")
print("Inputs verified; output directory:", OUTPUT_DIR)


In [3]:
def load_input(path):
    wb = load_workbook(path)
    ws = wb.active
    headers = [c.value for c in ws[1]]
    required = ["Churn-Influencing Feature", "Sign", "Actionable", "Intervention Variable", "Recommended Action", "Rationale", "Global Rank"]
    for col in required:
        if col not in headers:
            raise ValueError(f"Missing column {col} in {path}")
    idx = {name: headers.index(name) for name in headers}
    rows = [{h: ws.cell(r, j+1).value for j, h in enumerate(headers)} for r in range(2, ws.max_row+1)
            if ws.cell(r, idx["Churn-Influencing Feature"]+1).value is not None]
    return wb, ws, idx, rows

inputs = {key: load_input(DATA_DIR / FILES[key]) for key in ("global", "path")}
original_ws = load_workbook(DATA_DIR / FILES["original"], read_only=True, data_only=True).worksheets[0]
encoded_ws = load_workbook(DATA_DIR / FILES["encoded"], read_only=True, data_only=True).worksheets[0]
original_cols = list(next(original_ws.values))
encoded_cols = list(next(encoded_ws.values))
print("Original columns:", original_cols)
print("Encoded columns:", encoded_cols)
for key, (_, _, _, rows) in inputs.items():
    print(key, "rows:", len(rows))


Original columns: ['CustomerID', 'Gender', 'SeniorCitizen', 'Partner', 'Dependents', 'Tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']
Encoded columns: ['Gender', 'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling', 'Tenure', 'MonthlyCharges', 'TotalCharges', 'MultipleLines_No_phone_service', 'MultipleLines_Yes', 'InternetService_Fiber_optic', 'InternetService_No', 'OnlineSecurity_No_internet_service', 'OnlineSecurity_Yes', 'OnlineBackup_No_internet_service', 'OnlineBackup_Yes', 'DeviceProtection_No_internet_service', 'DeviceProtection_Yes', 'TechSupport_No_internet_service', 'TechSupport_Yes', 'StreamingTV_No_internet_service', 'StreamingTV_Yes', 'StreamingMovies_No_internet_service', 'StreamingMovies_Yes', 'Contract_Month-to-month', 'Contract_One_year', 'PaymentMethod_Bank

In [4]:
# Dataset-derived schema. Avoid sending customer-level records to the API.
# Categorical options are taken from the ORIGINAL dataset, not guessed from dummy names.
NUMERIC = {"Tenure", "MonthlyCharges", "TotalCharges"}
EXCLUDE = {"CustomerID", "Churn"}
values = defaultdict(set)
for row in list(original_ws.values)[1:]:
    for col, val in zip(original_cols, row):
        if col not in NUMERIC | EXCLUDE and val is not None:
            values[col].add(str(val))
category_schema = {k: sorted(v) for k, v in values.items()}
# Binary indicators can be encoded without an underscore (e.g. PhoneService).
# A dummy feature maps to the longest matching original feature prefix.
def source_variable(feature):
    if feature in original_cols:
        return feature
    matches = [c for c in original_cols if feature.startswith(c + "_")]
    if not matches:
        raise ValueError(f"Cannot map encoded feature: {feature}")
    return max(matches, key=len)

def normalize_category(value):
    return re.sub(
        r"[^a-zA-Z0-9]+",
        "_",
        str(value).strip()
    ).strip("_").lower()

def validate_source_feature(feature, original_ws, encoded_cols):
    """
    Validate a Source feature against:
    - Encoded dataset columns
    - Original dataset columns
    - Original categorical values, including omitted
      one-hot reference categories
    """

    # Read original column names
    headers = [
        str(cell.value).strip()
        for cell in original_ws[1]
        if cell.value is not None
    ]

    # 1. Feature exists in encoded dataset
    if feature in encoded_cols:
        return True

    # 2. Feature exists directly in original dataset
    if feature in headers:
        return True

    # 3. Feature represents an original categorical value
    for col_idx, col_name in enumerate(headers):

        prefix = f"{col_name}_"

        if feature.startswith(prefix):

            category = feature[len(prefix):]

            valid_categories = {
                normalize_category(row[col_idx])
                for row in original_ws.iter_rows(
                    min_row=2,
                    values_only=True
                )
                if row[col_idx] is not None
            }

            if normalize_category(category) in valid_categories:
                return True

    return False

pairs = {}
for key, (_, _, _, rows) in inputs.items():
    for row in rows:
    
        feature = str(
            row["Churn-Influencing Feature"]
        ).strip()
    
        sign = str(row["Sign"]).strip()
    
        if not validate_source_feature(
            feature,
            original_ws,
            encoded_cols
        ):
            raise ValueError(
                f"Source cannot be mapped to original dataset: {feature}"
            )
        if sign not in ("+", "-"):
            raise ValueError(f"Unexpected sign for {feature}: {sign}")
        if feature in pairs and pairs[feature]["sign"] != sign:
            raise ValueError(f"Conflicting signs across workbooks: {feature}")
        pairs[feature] = {"feature": feature, "sign": sign, "original_variable": source_variable(feature)}

payload = {
    "features": list(pairs.values()),
    "original_categories": category_schema,
    "numeric_original_variables": sorted(NUMERIC),
    "encoded_feature_names": [str(c) for c in encoded_cols if c not in ("CustomerID", "Churn")],
    "dataset_context": "Telco Customer Churn; signs and priorities were supplied by causal analysis; do not infer additional edges from data."
}
print("Unique features:", len(pairs))
print(json.dumps(payload["features"], indent=2))


Unique features: 15
[
  {
    "feature": "Contract_Month-to-month",
    "sign": "+",
    "original_variable": "Contract"
  },
  {
    "feature": "InternetService_Fiber_optic",
    "sign": "+",
    "original_variable": "InternetService"
  },
  {
    "feature": "MonthlyCharges",
    "sign": "+",
    "original_variable": "MonthlyCharges"
  },
  {
    "feature": "OnlineBackup_Yes",
    "sign": "-",
    "original_variable": "OnlineBackup"
  },
  {
    "feature": "PhoneService",
    "sign": "-",
    "original_variable": "PhoneService"
  },
  {
    "feature": "MultipleLines_No_phone_service",
    "sign": "+",
    "original_variable": "MultipleLines"
  },
  {
    "feature": "InternetService_No",
    "sign": "+",
    "original_variable": "InternetService"
  },
  {
    "feature": "PaymentMethod_Mailed_check",
    "sign": "-",
    "original_variable": "PaymentMethod"
  },
  {
    "feature": "MultipleLines_Yes",
    "sign": "-",
    "original_variable": "MultipleLines"
  },
  {
    "feature": "Con

## Intervention formulation prompt

This is an explicitly versioned prompt for **new API-based generation**. The API receives **only the schema and causal feature/sign pairs**, not customer records. Any category-specific causal justification must be supported by the provided signs; no invented edge weights or significance claims are allowed.

In [ ]:
SYSTEM_PROMPT = """You are a careful research assistant formulating candidate customer-retention interventions from observational causal-discovery outputs.
For every feature/sign pair return exactly one structured recommendation.
Rules:
1. Interpret feature names using the supplied original dataset categories and encoded feature list.
2. A positive sign (+) means the learned relationship with churn is positive; a negative sign (-) means it is negative. These are not established intervention effects.
3. For numerical controllable variables, propose an operationally feasible direction; for negative signs, maintain or encourage the protective state where reasonable.
4. For one-hot indicators, name the original categorical variable as intervention_variable, never the dummy column. Select ONLY valid original category values.
5. Avoid impossible or inconsistent service combinations. In particular, 'No phone service' and 'No internet service' indicators are dependent states, not independently modifiable features; mark those indicators Actionable NO and, if appropriate, mention the underlying service change as a candidate.
6. Distinguish directly controllable customer/service settings from derived or historical values (e.g., tenure, total charges). If not directly actionable, say NO.
7. If the evidence does not identify which alternative category reduces churn, explicitly mark the recommendation as a candidate requiring validation. Do not infer that the unobserved category has a negative causal edge.
8. For positive binary service features, suggest reducing or disabling the feature only if operationally coherent; for negative features, enabling/maintaining may be appropriate.
9. Do not invent causal edges, effect sizes, statistical significance, cost estimates, or treatment effectiveness.
10. Rankings and importance scores are intentionally omitted: generate the same recommendation for the same feature and sign regardless of ranking method.
11. Keep rationale concise, evidence-bounded, and explicit about uncertainty.
12. Preserve feature strings exactly and produce exactly one item for each input feature.
"""
USER_PROMPT = "Generate recommendations from this JSON schema and causal evidence:\n" + json.dumps(payload, ensure_ascii=False, sort_keys=True)
PROMPT_SHA256 = hashlib.sha256((SYSTEM_PROMPT + USER_PROMPT).encode()).hexdigest()
print("Prompt SHA-256:", PROMPT_SHA256)


In [ ]:
# Explicit JSON schema for reliable machine-readable outputs
ITEM_SCHEMA = {
    "type": "object", "additionalProperties": False,
    "properties": {
        "feature": {"type": "string"},
        "actionable": {"type": "string", "enum": ["YES", "NO"]},
        "intervention_variable": {"type": "string"},
        "recommended_action": {"type": "string"},
        "rationale": {"type": "string"}
    },
    "required": ["feature", "actionable", "intervention_variable", "recommended_action", "rationale"]
}
OUTPUT_SCHEMA = {
    "name": "churn_intervention_recommendations", "strict": True,
    "schema": {"type": "object", "additionalProperties": False,
               "properties": {"recommendations": {"type": "array", "items": ITEM_SCHEMA}},
               "required": ["recommendations"]}
}

api_key = os.getenv("OPENAI_API_KEY") or getpass("OpenAI API key (hidden): ")

client = OpenAI(api_key=api_key, timeout=180.0, max_retries=2)

# Remove the key from the notebook namespace after initializing the client.
del api_key
RUN_API = True  
if not RUN_API:
    print("Dry-run: API call disabled. Set RUN_API=True to execute the next cell.")

In [7]:
if not RUN_API:
    raise RuntimeError("Set RUN_API=True in the preceding cell, then rerun.")
started_at = datetime.now(timezone.utc).isoformat()
# Responses API: structured JSON output. Do not send temperature or seed without
# verifying support for this model.
try:
    response = client.responses.create(
        model=MODEL,
        instructions=SYSTEM_PROMPT,
        input=USER_PROMPT,
        text={"format": {"type": "json_schema", **OUTPUT_SCHEMA}},
        store=False,
    )
except Exception as exc:
    raise RuntimeError(
        "The selected model could not complete a Responses API request with "
        "JSON Schema structured output. Check model support and API access; "
        "no fallback model has been used."
    ) from exc
raw_text = response.output_text
if not raw_text:
    raise RuntimeError(f"No text returned; response status: {response.status}")
result = json.loads(raw_text)
actual_model = response.model
print("Actual model:", actual_model, "status:", response.status)
print("Tokens:", response.usage)


Actual model: gpt-6-astra status: completed
Tokens: ResponseUsage(input_tokens=1295, input_tokens_details=InputTokensDetails(cache_write_tokens=1292, cached_tokens=0, cache_write_tokens_details=InputTokensDetailsCacheWriteTokensDetails(ttl_12h_tokens=0, ttl_30m_tokens=1292)), output_tokens=1455, output_tokens_details=OutputTokensDetails(reasoning_tokens=79), total_tokens=2750)


In [8]:
# Strict validation BEFORE writing any workbook.
items = result["recommendations"]
if len(items) != len(pairs):
    raise ValueError(f"Expected {len(pairs)} recommendations, got {len(items)}")
by_feature = {}
for item in items:
    f = item["feature"]
    if f not in pairs or f in by_feature:
        raise ValueError(f"Unexpected/duplicate feature: {f}")
    if item["actionable"] not in ("YES", "NO"):
        raise ValueError(f"Invalid actionable value: {f}")
    if item["intervention_variable"] != pairs[f]["original_variable"]:
        raise ValueError(f"Intervention variable mismatch: {f}")
    if not item["recommended_action"].strip() or not item["rationale"].strip():
        raise ValueError(f"Missing action or rationale: {f}")
    # If a recommendation specifies a categorical assignment, validate its category.
    action = item["recommended_action"].strip()
    var = item["intervention_variable"]
    if "=" in action and var in category_schema:
        left, right = action.split("=", 1)
        if left.strip() == var:
            target = right.strip().removesuffix("(candidate)").strip()
            if target not in category_schema[var]:
                raise ValueError(f"Invalid category for {f}: {target!r}; valid: {category_schema[var]}")
    by_feature[f] = item

# Explicit semantic checks requiring human review (do not silently rewrite output).
review_flags = []
for f, item in by_feature.items():
    if f.endswith("_No_phone_service") or f.endswith("_No_internet_service"):
        if item["actionable"] != "NO":
            review_flags.append(f"Dependent indicator should be NO: {f}")
    if "candidate" in item["recommended_action"].lower():
        review_flags.append(f"Candidate requires empirical validation: {f}")
print("Validated", len(by_feature), "recommendations")
print("Manual review flags:", *review_flags, sep="\n - ")


Validated 15 recommendations
Manual review flags:
 - Candidate requires empirical validation: Contract_Month-to-month
 - Candidate requires empirical validation: PaymentMethod_Mailed_check


In [ ]:
# Fill only the four blank LLM columns; keep all input values, ranks, scores and formatting.
from openpyxl.styles import Alignment
saved = {}
for key, (wb, ws, idx, rows) in inputs.items():
    for row_num in range(2, ws.max_row + 1):
        feature = ws.cell(row_num, idx["Churn-Influencing Feature"]+1).value
        if feature is None:
            continue
        item = by_feature[str(feature)]
        assignments = {
            "Actionable": item["actionable"],
            "Intervention Variable": item["intervention_variable"],
            "Recommended Action": item["recommended_action"],
            "Rationale": item["rationale"],
        }
        for col, val in assignments.items():
            cell = ws.cell(row_num, idx[col]+1)
            if cell.value not in (None, ""):
                raise ValueError(f"Refusing to overwrite existing value: {key} {cell.coordinate}")
            cell.value = val
            cell.alignment = Alignment(vertical="top", wrap_text=True)
    filename = Path(FILES[key]).stem + " - API completed.xlsx"
    dest = OUTPUT_DIR / filename
    wb.save(dest)
    saved[key] = str(dest)

# Preserve the raw response and all methodological metadata for audit/reproduction.
metadata = {
    "timestamp_utc": started_at,
    "requested_model": MODEL, "returned_model": actual_model,
    "temperature": "not specified", "top_p": "not specified", "seed": "not specified",
    "api_endpoint": "responses.create",
    "response_id": response.id,
    "system_fingerprint": getattr(response, "system_fingerprint", None),
    "response_status": response.status,
    "usage": response.usage.model_dump() if response.usage else None,
    "prompt_sha256": PROMPT_SHA256,
    "system_prompt": SYSTEM_PROMPT,
    "user_prompt": USER_PROMPT,
    "output_schema": OUTPUT_SCHEMA,
    "raw_response": result,
    "output_files": saved,
    "manual_review_status": "NOT PERFORMED — requires researcher verification",
    "review_flags": review_flags,
}
audit_path = OUTPUT_DIR / "llm_run_audit.json"
audit_path.write_text(json.dumps(metadata, indent=2, ensure_ascii=False), encoding="utf-8")
for key, p in saved.items():
    check = load_workbook(p, read_only=True, data_only=True).active
    print(key, "saved", p, "rows", check.max_row)
print("Audit:", audit_path)
